In [21]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from tqdm import tqdm
import os
from PIL import Image
import io
import warnings
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_vl/redline_vl_attr_sample_251107_train_v2_vlsft_rm01_p3n1.json"
df = pd.read_json(json_file_path)

def is_valid_image(image_path):
    try:
        with Image.open(image_path) as img:
            img.verify()  # 验证图像完整性
        return True
    except Exception as e:
        warnings.warn(f"Invalid image {image_path}: {str(e)}")
        return False

# 检查DataFrame的基本信息
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# 检查是否存在images字段
if 'images' not in df.columns:
    print("Warning: 'images' column not found in DataFrame!")
    # 尝试查找可能的图片字段
    possible_image_cols = [col for col in df.columns if 'image' in col.lower() or 'img' in col.lower()]
    print(f"Possible image columns: {possible_image_cols}")
else:
    print(f"'images' column found. Data type: {df['images'].dtype}")
    
    # 检查images字段的前几个样本
    print("\nFirst few 'images' field samples:")
    for i in range(min(5, len(df))):
        sample = df.iloc[i]['images']
        print(f"Sample {i}: {type(sample)} - {sample}")

    # 提取所有图片地址
    all_image_paths = []
    
    # 遍历DataFrame中的每一行
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting image paths"):
        images_field = row['images']
        
        # 处理不同类型的images字段
        if isinstance(images_field, list):
            # 如果是列表，直接扩展
            all_image_paths.extend(images_field)
        elif isinstance(images_field, str):
            # 如果是字符串，尝试解析为JSON列表
            try:
                images_list = json.loads(images_field)
                if isinstance(images_list, list):
                    all_image_paths.extend(images_list)
                else:
                    print(f"Warning: Row {idx} - 'images' field is string but not a valid JSON list")
            except json.JSONDecodeError:
                print(f"Warning: Row {idx} - 'images' field is string but cannot be parsed as JSON")
        elif pd.isna(images_field) or images_field is None:
            print(f"Warning: Row {idx} - 'images' field is empty/None")
        else:
            print(f"Warning: Row {idx} - 'images' field has unexpected type: {type(images_field)}")

    # 去重
    unique_image_paths = list(set(all_image_paths))
    
    print(f"\nTotal image paths extracted: {len(all_image_paths)}")
    print(f"Unique image paths: {len(unique_image_paths)}")
    print(f"First 10 unique image paths:")
    for i, path in enumerate(unique_image_paths[:10]):
        print(f"{i+1}: {path}")

    # 可选：保存图片路径到文件
    output_dir = os.path.dirname(json_file_path)
    image_paths_file = os.path.join(output_dir, "extracted_image_paths.txt")
    
    a = unique_image_paths
    unique_image_paths = []
    with open(image_paths_file, 'w') as f:
        for path in a:
            unique_image_paths.append(f"/Users/liyong.1024/Workspace/xz-shield/{path}")
            f.write(f"/Users/liyong.1024/Workspace/xz-shield/{path}\n")
    
    print(f"\nImage paths saved to: {image_paths_file}")
    x = unique_image_paths[0]
    print(f"{len(unique_image_paths)}\n{unique_image_paths[0]}\n{os.path.exists(x)}\n{is_valid_image(x)}")

    # 可选：创建包含所有图片信息的新DataFrame
    image_df = pd.DataFrame({
        'image_path': unique_image_paths,
        'file_name': [os.path.basename(path) for path in unique_image_paths],
        'file_dir': [os.path.dirname(path) for path in unique_image_paths]
    })

        
    # 检查文件是否存在
    image_df['file_exists'] = image_df['image_path'].apply(
        lambda x: is_valid_image(x) if isinstance(x, str) else False
    )
    
    print(f"\nImage DataFrame shape: {image_df.shape}")
    print(f"Existing files: {image_df['file_exists'].sum()}/{len(image_df)}")
    
    # 保存image_df到parquet文件（如果需要）
    # parquet_file = os.path.join(output_dir, "image_metadata.parquet")
    # if ARROW_AVAILABLE:
    #     table = pa.Table.from_pandas(image_df)
    #     pq.write_table(table, parquet_file)
    #     print(f"Image metadata saved to Parquet: {parquet_file}")




DataFrame shape: (4143, 5)
Columns: ['instruction', 'input', 'output', 'system', 'images']
'images' column found. Data type: object

First few 'images' field samples:
Sample 0: <class 'list'> - ['./.cache/mm01_251028/sku-jfs-t1-238818-7-24200-307944-6756d2d7F1cd907b7-2d4fffca72d2861f.jpg', './.cache/mm01_251028/sku-jfs-t1-246950-5-28472-123665-6756d308Fda3cdf31-c4186249fce75aca.jpg', './.cache/mm01_251028/sku-jfs-t1-197151-9-52811-179153-6756d308F13359d11-e5be2cd31ef9c140.jpg', './.cache/mm01_251028/sku-jfs-t1-246513-19-28797-73781-6756d308Fcb0c8d03-3351cbf43f7fe4df.jpg', './.cache/mm01_251028/sku-jfs-t1-239764-20-26639-326984-6756d308F3c9c43dc-4238aa4030313d46.jpg', './.cache/mm01_251028/sku-jfs-t1-226454-29-33052-87995-6756d308F2d8834fd-0a0a3c0dd2f6246b.jpg', './.cache/mm01_251028/sku-jfs-t1-242744-5-27861-49059-6756d308Fe3190dfc-8a15446cf50ed2a6.jpg', './.cache/mm01_251028/sku-jfs-t1-220795-15-51709-88277-6756d308Fd552685a-b161d769a1f98cd4.jpg', './.cache/mm01_251028/sku-jfs-t1-2494

Extracting image paths: 100%|██████████| 4143/4143 [00:00<00:00, 89131.11it/s]


Total image paths extracted: 37765
Unique image paths: 24412
First 10 unique image paths:
1: ./.cache/mm01_251028/cms-jfs-t1-188667-32-15737-358982-61022357E60c1b306-9b2ea142ec8df1cd.jpg
2: ./.cache/mm01_251028/imgzone-jfs-t1-280840-18-59-47583-67ce9530Fc603234d-04e1fcfd109a9b39.jpg
3: ./.cache/mm01_251028/imgzone-jfs-t1-25323-24-22290-134805-66ac8f7fF61b42709-49ae791a5ce64220.jpg
4: ./.cache/mm01_251028/img-jfs-t1-302856-27-416-38461-680f4f1fFd990f753-9c651d28a9a52f71.jpg
5: ./.cache/mm01_251028/img-jfs-t1-133040-40-43871-47797-66dfff1bF9fab2c13-a5255833742fc7d1.jpg
6: ./.cache/mm01_251028/img-jfs-t1-334410-14-21089-113099-68e7ce62F0f90bd62-087be299033f977b.jpg
7: ./.cache/mm01_251028/sku-jfs-t1-341741-29-1811-78975-68c13f29F3b9fa144-f057613013d80f1d.jpg
8: ./.cache/mm01_251028/sku-jfs-t1-327064-8-1694-125732-68940b43Fb7c6473f-ea1318c9afd31fe8.jpg
9: ./.cache/mm01_251028/sku-jfs-t1-309305-33-19119-34872-6880bff5F1ab44f5b-98ea5e7cf36ab633.jpg
10: ./.cache/mm01_251028/img-jfs-t1-274400


Image DataFrame shape: (24412, 4)
Existing files: 24412/24412


In [5]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v1_p1n02_vlsft_rq01.json"
output_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v1_p2n02_vlsft_rq01.json"
df = pd.read_json(json_file_path)

pos_data = df[(df['output'] == '是')]
neg_data = df[(df['output'] == '否')]
df = pd.concat([pos_data]*2 + [neg_data], ignore_index=True)

df.to_json(output_file_path, orient='records', date_format='iso', force_ascii=False, indent=0,lines=True)




In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_val_v1_vlsft_rq01.json"
pos_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_val_v1_vlsft_rq01_pos.json"
neg_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_val_v1_vlsft_rq01_neg.json"
df = pd.read_json(json_file_path)

pos_data = df[(df['output'] == '是')]
neg_data = df[(df['output'] == '否')]


pos_data.to_json(pos_file_path, orient='records', date_format='iso', force_ascii=False, indent=2)
neg_data.to_json(neg_file_path, orient='records', date_format='iso', force_ascii=False, indent=2)




In [2]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
json_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v1.xlsx"
train_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v1_p1n02.xlsx"
val_file_path = "/Users/liyong.1024/Documents/protect/redline_platform/redline_platform_sample_251112_train_v1_p0n08.xlsx"
df = pd.read_excel(json_file_path)
# 设置随机种子
np.random.seed(42)
df_shuffled = df.sample(frac=1).reset_index(drop=True)
train_num = int(len(df_shuffled) * 0.9)
df_train = df_shuffled[:train_num]
df_val = df_shuffled[train_num:]

# pos_data = df[(df['redline_label'] == 1)]
# neg_data = df[(df['redline_label'] == 0)]
# train_num = int(len(neg_data) * 0.2)
# df_train = pd.concat([pos_data] + [neg_data[:train_num]], ignore_index=True)
# df_val = pd.concat([neg_data[train_num:]], ignore_index=True)

df_train.to_excel(train_file_path)
df_val.to_excel(val_file_path)
# df.to_json(output_file_path, orient='records', date_format='iso', force_ascii=False, indent=2)



In [9]:
# print(df[:1]["prompt"].values)
print("{%- if tools %}\n    {{- '<|im_start|>system\\n' }}\n    {%- if messages[0].role == 'system' %}\n        {%- if messages[0].content is string %}\n            {{- messages[0].content }}\n        {%- else %}\n            {%- for content in messages[0].content %}\n                {%- if 'text' in content %}\n                    {{- content.text }}\n                {%- endif %}\n            {%- endfor %}\n        {%- endif %}\n        {{- '\\n\\n' }}\n    {%- endif %}\n    {{- \"# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0].role == 'system' %}\n        {{- '<|im_start|>system\\n' }}\n        {%- if messages[0].content is string %}\n            {{- messages[0].content }}\n        {%- else %}\n            {%- for content in messages[0].content %}\n                {%- if 'text' in content %}\n                    {{- content.text }}\n                {%- endif %}\n            {%- endfor %}\n        {%- endif %}\n        {{- '<|im_end|>\\n' }}\n    {%- endif %}\n{%- endif %}\n{%- set image_count = namespace(value=0) %}\n{%- set video_count = namespace(value=0) %}\n{%- for message in messages %}\n    {%- if message.role == \"user\" %}\n        {{- '<|im_start|>' + message.role + '\\n' }}\n        {%- if message.content is string %}\n            {{- message.content }}\n        {%- else %}\n            {%- for content in message.content %}\n                {%- if content.type == 'image' or 'image' in content or 'image_url' in content %}\n                    {%- set image_count.value = image_count.value + 1 %}\n                    {%- if add_vision_id %}Picture {{ image_count.value }}: {% endif -%}\n                    <|vision_start|><|image_pad|><|vision_end|>\n                {%- elif content.type == 'video' or 'video' in content %}\n                    {%- set video_count.value = video_count.value + 1 %}\n                    {%- if add_vision_id %}Video {{ video_count.value }}: {% endif -%}\n                    <|vision_start|><|video_pad|><|vision_end|>\n                {%- elif 'text' in content %}\n                    {{- content.text }}\n                {%- endif %}\n            {%- endfor %}\n        {%- endif %}\n        {{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"assistant\" %}\n        {{- '<|im_start|>' + message.role + '\\n' }}\n        {%- if message.content is string %}\n            {{- message.content }}\n        {%- else %}\n            {%- for content_item in message.content %}\n                {%- if 'text' in content_item %}\n                    {{- content_item.text }}\n                {%- endif %}\n            {%- endfor %}\n        {%- endif %}\n        {%- if message.tool_calls %}\n            {%- for tool_call in message.tool_calls %}\n                {%- if (loop.first and message.content) or (not loop.first) %}\n                    {{- '\\n' }}\n                {%- endif %}\n                {%- if tool_call.function %}\n                    {%- set tool_call = tool_call.function %}\n                {%- endif %}\n                {{- '<tool_call>\\n{\"name\": \"' }}\n                {{- tool_call.name }}\n                {{- '\", \"arguments\": ' }}\n                {%- if tool_call.arguments is string %}\n                    {{- tool_call.arguments }}\n                {%- else %}\n                    {{- tool_call.arguments | tojson }}\n                {%- endif %}\n                {{- '}\\n</tool_call>' }}\n            {%- endfor %}\n        {%- endif %}\n        {{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}\n        {%- if loop.first or (messages[loop.index0 - 1].role != \"tool\") %}\n            {{- '<|im_start|>user' }}\n        {%- endif %}\n        {{- '\\n<tool_response>\\n' }}\n        {%- if message.content is string %}\n            {{- message.content }}\n        {%- else %}\n            {%- for content in message.content %}\n                {%- if content.type == 'image' or 'image' in content or 'image_url' in content %}\n                    {%- set image_count.value = image_count.value + 1 %}\n                    {%- if add_vision_id %}Picture {{ image_count.value }}: {% endif -%}\n                    <|vision_start|><|image_pad|><|vision_end|>\n                {%- elif content.type == 'video' or 'video' in content %}\n                    {%- set video_count.value = video_count.value + 1 %}\n                    {%- if add_vision_id %}Video {{ video_count.value }}: {% endif -%}\n                    <|vision_start|><|video_pad|><|vision_end|>\n                {%- elif 'text' in content %}\n                    {{- content.text }}\n                {%- endif %}\n            {%- endfor %}\n        {%- endif %}\n        {{- '\\n</tool_response>' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}\n            {{- '<|im_end|>\\n' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- '<|im_start|>assistant\\n' }}\n{%- endif %}\n")

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {%- if messages[0].content is string %}
            {{- messages[0].content }}
        {%- else %}
            {%- for content in messages[0].content %}
                {%- if 'text' in content %}
                    {{- content.text }}
                {%- endif %}
            {%- endfor %}
        {%- endif %}
        {{- '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if me

In [10]:
redline_data = df[(df['redline_label'] == 1)]
tp_data = df[(df['answer_label'] == 1) & (df['answer_pred_p13'] == 1)].sample(frac=0.1)
fp_data = df[(df['answer_label'] == 1) & (df['answer_pred_p13'] == 0)].sample(frac=0.5)
new_df = pd.concat([redline_data] + [fp_data] + [tp_data], ignore_index=True)
new_df.to_excel("/Users/liyong.1024/Documents/protect/redline_database/redline_annotation_0904_merge_v3_filter.xlsx")

In [11]:
new_df

,index,send_time,session_id,message_id,bot_id,user_pin,上文对话,用户问,答案,候选知识,...,redline_label,answer_label,reject,source_file,redline_pred_p13,answer_pred_p13,no_rate,yes_rate,llm_response,rand
0,0,2025-08-14 18:43:33.115,df72d1f5ccfac5b4fa3805387787b6ff,A6A9A135-3973-4B4C-B8D5-30309CDA0406,23489,13170971432_p,"""用户: 顾客加入咨询 订单ID:319132805296\\n\n用户: 我想问下\\n\...",这个手表拦截的电话号码怎么看不到啊,"{""answer"":[{""content"":""您好，米兔儿童电话手表7X支持通话白名单功能，...",[1] 有什么功能\n<strong>米兔儿童电话手表7X</strong><br/>支持通...,...,1,0,False,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
1,42,2025-08-14 20:08:47.111,5f857dfd-2f96-4105-a7f5-b93cdab5e652,MasterAgentXZ_7e375630ae214163a39a3e1d2807f080,1081088,jd_zfpuvrobtsoh,"""用户: 顾客jd_zfpuvrobtsoh加入咨询""}",如何参加国补？,"{""answer"":[{""content"":""参加国补的具体步骤如下：1. 确认您购买的产品...","""}",...,1,0,True,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
2,134,2025-08-14 20:44:32.825,70fb08082185fe4a4913396f2dee4ad4,600D8474-6676-41CC-8AC9-C47E2B0AC4CD,23489,jd_58e84ed2a2308,"""用户: 有没有吸力25立方的\\n\n客服: 米家智能欧式吸油烟灶套装S2（米家智能欧式吸...",P2是最新款吗,"{""answer"":[{""content"":""米家智能净烟机P2是小米的最新款产品之一，具备...","[1] 是否新款\n<p><span style=\\""color: rgb(192, 0,...",...,1,0,True,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
3,182,2025-08-14 22:41:20.846,8d0098a83ac0c02674c4f32a94aa5fc9,0cb8dbcf-5257-1ffa-28a5-92daf244a545,132483,jd_4aa5ebdd36aaa,"""用户: 顾客jd_4aa5ebdd36aaa加入咨询，商品编号：100171882477\...",我娃3个多月现在15斤 L会不会小呀 肚子上很多肉肉,"{""answer"":[{""content"":""根据您的描述，L码适合7-12个月，体重9-1...",[1] L码多大宝宝用\n根据您的描述推荐：L码<br/>#E-s01适合月龄：7-12月（...,...,1,0,False,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
4,216,2025-08-14 07:57:45.402,6ff12fd4f539d1e4917b19bfb504242c,facf35054cb64fbcb371b681d75ad86c,132,jd_5472e0e9e62eb,"""用户: 顾客jd_5472e0e9e62eb加入咨询 商品：100183021313\\\...",要底座充电吗 支持快充吗,"{""answer"":[{""content"":""vivo WATCH 5手表仅支持通过配置的充...",[1] 供电方式及充电器参数\nvivo WATCH 5手表仅支持<span style=\...,...,1,0,False,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3227,7540,2025-08-25T18:35:01.368000,8901114afca57378b8e0211b3c7e264b,36d2db4556434e548d18c3f1b066d7b6,23489,jd_5ab4f51872e33,"{""formatted_context"": ""用户: 顾客jd_5ab4f51872e33加...",怎么使用,"{""answer"":[{""content"":""米家智能腰部按摩仪EMS版<br/>【使用说明...",[1] 商品使用方法\n米家智能腰部按摩仪EMS版<br/>【使用说明】?【<a href=...,...,0,1,False,redline_annotation_0826.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.531141
3228,8279,2025-08-25T06:11:49.396000,c65ac20668b4b310ddb1194a4c5c1e13,72e6ec8543ef48579f8086ecbfb1a23d,23489,jd_wepmiufjulib,"{""formatted_context"": ""用户: 顾客jd_wepmiufjulib加入...",什么是玻纤材质什么是玻纤材质？？,"{""answer"":[{""content"":""玻纤材质是指玻璃纤维增强塑料（Glass Fi...",[1] 名词释义\n商品功能、名词说明可以参考商品详情页哦~\n\n[2] 商品材质\n<s...,...,0,1,False,redline_annotation_0826.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.088937
3229,6031,2025-08-21T16:11:04.909000,23dfe947562337fab9935a62684d94e1,6498AA2C-83BD-419B-815F-D77ED259F55F,543317,jd_64a6c25cc9c37,"""用户: https://item.jd.com/10156627110570.html?...",16.8/19.9/59.9什么区别 看外观都一样,"{""answer"":[{""content"":""您好，关于不同款型商品对比，您可以：1、对比详...",[1] 商品区别\n尊敬的顾客您好！感谢信任选择本店。<br/>关于不同款型商品对比，您可以...,...,0,1,False,redline_annotation_0822.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.653778
3230,4770,2025-08-21T09:02:26.040000,bcd20308b4f99d72e38e8c1bd7da61a0,bdeeb296-64ad-bf1a-7493-8ee5485a95b3,769654,jd_WDmDDZttisHQ,"""用户: 顾客jd_WDmDDZttisHQ加入咨询，商品编号：1011530652386...",是真品吗,"{""answer"":[{""content"":""店铺所售商品均为正品行货，商品都是通过正规渠道...",[1] 正品保障\n店铺所售商品均为正品行货，商品都是通过正规渠道进货，商品质量都经过严格的...,...,0,1,False,redline_annotation_0822.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.017648


In [43]:
import json

request = df.loc[1]["request"]
response = df.loc[1]["response"]
def parse_request(request):
    try:
        # print(request)
        req_json = json.loads(request) 

        # print(req_json)
        return req_json
    except:
        print(f"error:{request}")
        return {}


def parse_response(response):
    try:
        response = f"{response}"
        response = response.replace("\\\\", "\\")
        # print(response)
        resp_json = json.loads(response) 
        print(resp_json)
        resp_json = json.loads(resp_json) 
        print(resp_json["reject"])
        if "reject" not in  resp_json:
             resp_json["reject"] = False
        return resp_json
    except:
        print(f"error:{response}")
        return {"reject":False,"reason":""}

parse_response(response)
parse_request(request)


{"reject":true,"reason":"<think>\n\n</think>\n\n{\"一级\":\"事实错误\",\"二级\": \"尺码问题\"}"}
True


{'answer': [{'content': '', 'type': 'text'}],
 'context': [{'content': '160斤身高1.7 穿多大的号',
   'intent': 'knowledge_answer',
   'role': 'user'},
  {'content': '您好，根据您的身高和体重，建议您选择适合您的尺码。您可以参考店铺内的尺码表，选择最适合您的尺码。如果您有具体的产品型号或系列名称，可以告诉我，我可以为您提供更详细的建议。;',
   'role': 'waiter'},
  {'content': 'https://item.jd.com/10151633846635.html?sdx=ehi-lLxFuZiE6JnJZIJaj8UotDaXCw4rsmpEuKhAY9iCPe_RLJhZ53rsrUrhUmGZ\\t 商品ID：10151633846635，商品名称：YOUNG VERSACE范思新款哲男士夏季新款T恤圆领潮流短袖百搭印花半袖时尚男士 黑色 范思哲 XL 范思哲',
   'role': 'user'},
  {'content': '很高兴为您服务，选择这款商品的您超有眼光，请问您要咨询这款产品什么问题呢？', 'role': 'waiter'}],
 'knowledges': [{'answer': [{'content': '店铺有多款产品供您选择，所有系列都是不错的呢，您可以根据需求在店铺首页搜索选择的哦~',
     'type': 'text'}],
   'id': 1,
   'isPkAnswer': 1,
   'name': '尺码推荐',
   'score': 0.9512712,
   'type': 'faq_nlu_manager'},
  {'answer': [{'content': '您可以点击商品详情查看“规格与包装”里面的配件信息哦~', 'type': 'text'}],
   'id': 2,
   'isPkAnswer': 0,
   'name': '包装清单',
   'score': 0.33268708,
   'type': 'faq_nlu_manager'},
  {'answer': [{'content': '每个人

In [38]:

# 方法3: 使用assign方法（更简洁）
print("方法3: 使用assign方法")
# parsed_data = df['request'].apply(parse_request)
parsed_data2 = df['response'].apply(parse_response)
print(parsed_data2)
df_result3 = df.assign(
    # answer=parsed_data.apply(lambda x: x['answer']),
    # context=parsed_data.apply(lambda x: x['context']),
    # knowledges=parsed_data.apply(lambda x: x['knowledges']),
    # query=parsed_data.apply(lambda x: x['query']),
    reject=parsed_data2.apply(lambda x: x['reject']),
    reason=parsed_data2.apply(lambda x: x['reason']),
)

print(df_result3)


方法3: 使用assign方法
error:nan
0        {"reject":false,"reason":"<think>\n\n</think>\...
1        {"reject":true,"reason":"<think>\n\n</think>\n...
2        {"reject":false,"reason":"<think>\n\n</think>\...
3        {"reject":false,"reason":"<think>\n\n</think>\...
4        {"reject":false,"reason":"<think>\n\n</think>\...
                               ...                        
31424    {"reject":false,"reason":"<think>\n\n</think>\...
31425    {"reject":true,"reason":"<think>\n\n</think>\n...
31426    {"reject":true,"reason":"<think>\n\n</think>\n...
31427    {"reject":false,"reason":"<think>\n\n</think>\...
31428    {"reject":false,"reason":"<think>\n\n</think>\...
Name: response, Length: 31429, dtype: object


TypeError: string indices must be integers, not 'str'

In [ ]:

# 方法3: 使用assign方法（更简洁）
print("方法3: 使用assign方法")
# parsed_data = df['request'].apply(parse_request)
parsed_data2 = df['response'].apply(parse_response)
print(parsed_data2)
df_result3 = df.assign(
    # answer=parsed_data.apply(lambda x: x['answer']),
    # context=parsed_data.apply(lambda x: x['context']),
    # knowledges=parsed_data.apply(lambda x: x['knowledges']),
    # query=parsed_data.apply(lambda x: x['query']),
    reject=parsed_data2.apply(lambda x: x['reject']),
    reason=parsed_data2.apply(lambda x: x['reason']),
)

print(df_result3)


方法3: 使用assign方法
error:nan
0        {"reject":false,"reason":"<think>\n\n</think>\...
1        {"reject":true,"reason":"<think>\n\n</think>\n...
2        {"reject":false,"reason":"<think>\n\n</think>\...
3        {"reject":false,"reason":"<think>\n\n</think>\...
4        {"reject":false,"reason":"<think>\n\n</think>\...
                               ...                        
31424    {"reject":false,"reason":"<think>\n\n</think>\...
31425    {"reject":true,"reason":"<think>\n\n</think>\n...
31426    {"reject":true,"reason":"<think>\n\n</think>\n...
31427    {"reject":false,"reason":"<think>\n\n</think>\...
31428    {"reject":false,"reason":"<think>\n\n</think>\...
Name: response, Length: 31429, dtype: object


TypeError: string indices must be integers, not 'str'

In [ ]:

# 方法3: 使用assign方法（更简洁）
print("方法3: 使用assign方法")
# parsed_data = df['request'].apply(parse_request)
parsed_data2 = df['response'].apply(parse_response)
print(parsed_data2)
df_result3 = df.assign(
    # answer=parsed_data.apply(lambda x: x['answer']),
    # context=parsed_data.apply(lambda x: x['context']),
    # knowledges=parsed_data.apply(lambda x: x['knowledges']),
    # query=parsed_data.apply(lambda x: x['query']),
    reject=parsed_data2.apply(lambda x: x['reject']),
    reason=parsed_data2.apply(lambda x: x['reason']),
)

print(df_result3)


方法3: 使用assign方法
error:nan
0        {"reject":false,"reason":"<think>\n\n</think>\...
1        {"reject":true,"reason":"<think>\n\n</think>\n...
2        {"reject":false,"reason":"<think>\n\n</think>\...
3        {"reject":false,"reason":"<think>\n\n</think>\...
4        {"reject":false,"reason":"<think>\n\n</think>\...
                               ...                        
31424    {"reject":false,"reason":"<think>\n\n</think>\...
31425    {"reject":true,"reason":"<think>\n\n</think>\n...
31426    {"reject":true,"reason":"<think>\n\n</think>\n...
31427    {"reject":false,"reason":"<think>\n\n</think>\...
31428    {"reject":false,"reason":"<think>\n\n</think>\...
Name: response, Length: 31429, dtype: object


TypeError: string indices must be integers, not 'str'